# Aula 03 — Camada Bronze

## Objetivo

Nesta etapa vamos iniciar nosso pipeline de Engenharia de Dados.

A camada **Bronze** será responsável por receber os dados provenientes da fonte original, preservando-os o máximo possível antes das etapas de limpeza e transformação.

### Fluxo desta etapa

CSV → Python/Pandas → PostgreSQL → Schema Bronze → Tabela vendas

Nossa fonte será o arquivo `dados/vendas.csv`.

O destino será o banco PostgreSQL `db_vendas`, executado em um container Docker.

Ao final desta etapa, teremos os 1.000 registros da fonte armazenados na camada Bronze.

In [ ]:
# ==========================================================
# ETAPA 1 - PREPARAÇÃO DO AMBIENTE
# ==========================================================

# O psycopg2 é o driver que permite que programas Python
# se comuniquem com bancos de dados PostgreSQL.
#
# O parâmetro -binary facilita a instalação porque já fornece
# os componentes necessários compilados.

%pip install psycopg2-binary

In [ ]:
# ==========================================================
# ETAPA 2 - IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

# Pandas será utilizado para leitura e manipulação
# da nossa base de vendas.
import pandas as pd

# psycopg2 será responsável pela comunicação
# entre Python e PostgreSQL.
import psycopg2

In [ ]:
# ==========================================================
# ETAPA 3 - CONFIGURAÇÃO DO POSTGRESQL
# ==========================================================

# Importa a função getpass.
# Ela permite solicitar uma senha sem exibi-la
# na tela enquanto o usuário digita.
from getpass import getpass


# Endereço onde o PostgreSQL está disponível.
# Como estamos acessando o Docker pelo nosso computador,
# utilizamos localhost.
DB_HOST = "localhost"


# Porta externa configurada no docker-compose.yml.
#
# Nosso computador acessa a porta 5434,
# que é direcionada para a porta 5432
# do PostgreSQL dentro do container.
DB_PORT = 5434


# Nome do banco de dados que criamos.
DB_NAME = "db_vendas"


# Usuário utilizado para acessar o PostgreSQL.
DB_USER = "postgres"


# Solicita a senha no momento da execução.
#
# Diferente de:
# DB_PASSWORD = "postgres"
#
# a senha não ficará escrita no código
# e também não será exibida enquanto digitamos.
DB_PASSWORD = getpass("Digite a senha do PostgreSQL: ")

In [ ]:
# ==========================================================
# ETAPA 4 - TESTE DE CONEXÃO COM O POSTGRESQL
# ==========================================================

try:
    # Cria uma conexão com o PostgreSQL.
    #
    # A senha NÃO está escrita diretamente no código.
    # DB_PASSWORD contém temporariamente a senha que foi
    # digitada de forma oculta através do getpass().
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    # Se a conexão for estabelecida sem erros,
    # esta mensagem será exibida.
    print("✅ Conexão com PostgreSQL realizada com sucesso!")

    # Fecha a conexão após o teste.
    conexao.close()

except Exception as erro:

    # Caso a conexão falhe, informa que ocorreu um problema.
    print("❌ Não foi possível conectar ao PostgreSQL.")

    # Mostra apenas o tipo do erro.
    # Evitamos imprimir informações desnecessárias
    # sobre a conexão.
    print("Tipo do erro:", type(erro).__name__)

Boa prática de segurança: nunca devemos inserir senhas diretamente no código. Neste projeto didático utilizamos getpass() para solicitar a credencial durante a execução. Em projetos de produção, normalmente utilizamos variáveis de ambiente ou serviços específicos para gerenciamento de segredos.

## Etapa 5 — Criando o Schema Bronze

No PostgreSQL, um **database** pode conter vários **schemas**, e cada schema pode conter várias tabelas.

Neste projeto utilizaremos os schemas para separar as etapas do nosso pipeline de Engenharia de Dados:

db_vendas
- bronze → dados brutos provenientes da fonte
- silver → dados tratados e padronizados
- gold → dados preparados para análise e BI

Nesta etapa criaremos apenas o schema `bronze`.

A camada Bronze será responsável por armazenar os dados provenientes do arquivo CSV com o mínimo possível de transformação.

In [ ]:
# ==========================================================
# ETAPA 5 - CRIAÇÃO DO SCHEMA BRONZE
# ==========================================================

try:

    # Abre novamente uma conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    # O cursor é o objeto utilizado para enviar
    # comandos SQL ao PostgreSQL.
    cursor = conexao.cursor()

    # CREATE SCHEMA cria uma nova área lógica dentro
    # do banco de dados.
    #
    # IF NOT EXISTS evita erro caso o schema
    # bronze já tenha sido criado anteriormente.
    cursor.execute("""
        CREATE SCHEMA IF NOT EXISTS bronze;
    """)

    # commit() confirma a alteração no banco.
    # Sem o commit, a criação poderia não ser persistida.
    conexao.commit()

    print("✅ Schema bronze criado com sucesso!")

except Exception as erro:

    print("❌ Erro ao criar o schema bronze.")
    print("Tipo do erro:", type(erro).__name__)

finally:

    # Fecha o cursor caso ele tenha sido criado.
    if "cursor" in locals():
        cursor.close()

    # Fecha a conexão com o PostgreSQL.
    if "conexao" in locals():
        conexao.close()

## Etapa 6 — Preparação dos dados para a camada Bronze

Agora que o schema `bronze` existe, precisamos carregar nossa fonte de dados para dentro do PostgreSQL.

Nossa fonte é:

`dados/vendas.csv`

O objetivo será criar:

`bronze.vendas`

Antes de criar a tabela, vamos ler o CSV novamente e verificar suas colunas e tipos.

### Por que fazer essa verificação?

O Pandas e o PostgreSQL possuem sistemas de tipos diferentes.

Por exemplo:

- `int64` no Pandas pode ser armazenado como `INTEGER` no PostgreSQL.
- `float64` pode ser armazenado como `NUMERIC`.
- textos podem ser armazenados como `TEXT` ou `VARCHAR`.
- datas devem ser armazenadas como `DATE` ou `TIMESTAMP`.

Nesta etapa vamos conhecer os dados antes de definir a estrutura da tabela.

In [ ]:
# ==========================================================
# ETAPA 6 - LEITURA DOS DADOS DE ORIGEM
# ==========================================================

# read_csv() lê novamente nossa fonte de dados.
#
# O arquivo continua sendo nossa fonte original.
# Ainda não estamos realizando nenhuma transformação.
df = pd.read_csv("dados/vendas.csv")


# Exibe a quantidade de linhas e colunas carregadas.
#
# shape[0] = quantidade de linhas
# shape[1] = quantidade de colunas
print("Quantidade de linhas:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])


# Exibe os nomes das colunas.
#
# Isso será importante para definirmos posteriormente
# a estrutura da tabela no PostgreSQL.
print("\nColunas encontradas:")

for coluna in df.columns:
    print("-", coluna)


# Exibe os tipos identificados automaticamente
# pelo Pandas durante a leitura do CSV.
print("\nTipos identificados pelo Pandas:")

print(df.dtypes)

## Etapa 7 — Criação da tabela Bronze

Após analisar os tipos presentes na fonte, podemos criar nossa primeira tabela no PostgreSQL.

A tabela será:

`bronze.vendas`

Ela armazenará os registros provenientes do arquivo `vendas.csv`.

Nesta aula utilizaremos tipos adequados do PostgreSQL, como `INTEGER`, `DATE`, `TEXT` e `NUMERIC`.

Ainda não realizaremos limpeza, padronização ou regras de negócio. Essas operações serão responsabilidade da camada Silver.

In [ ]:
# ==========================================================
# ETAPA 7 - CRIAÇÃO DA TABELA BRONZE.VENDAS
# ==========================================================

try:

    # Abre uma conexão com nosso banco db_vendas.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    # Cria o cursor responsável por enviar
    # comandos SQL para o PostgreSQL.
    cursor = conexao.cursor()


    # Cria a tabela vendas dentro do schema bronze.
    #
    # IF NOT EXISTS evita erro caso a célula
    # seja executada novamente.
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS bronze.vendas (

            id_venda INTEGER,

            data_venda DATE,

            cliente TEXT,

            produto TEXT,

            categoria TEXT,

            regiao TEXT,

            quantidade INTEGER,

            valor_unitario NUMERIC(12,2),

            desconto_percentual INTEGER,

            valor_total NUMERIC(12,2),

            forma_pagamento TEXT
        );
    """)


    # Confirma a criação da tabela.
    conexao.commit()

    print("✅ Tabela bronze.vendas criada com sucesso!")


except Exception as erro:

    print("❌ Erro ao criar a tabela bronze.vendas.")
    print("Tipo do erro:", type(erro).__name__)


finally:

    # Fecha o cursor.
    if "cursor" in locals():
        cursor.close()

    # Fecha a conexão.
    if "conexao" in locals():
        conexao.close()

## Etapa 8 — Carga dos dados na camada Bronze

Agora que a estrutura `bronze.vendas` foi criada no PostgreSQL, podemos carregar os registros provenientes do arquivo CSV.

O fluxo será:

`vendas.csv → Pandas → Python → PostgreSQL → bronze.vendas`

Nesta etapa utilizaremos o comando SQL `INSERT`.

Cada linha do DataFrame será enviada para a tabela `bronze.vendas`.

Antes da carga, a tabela será esvaziada com `TRUNCATE` para permitir que esta célula seja executada novamente sem duplicar os registros.

Ao final, vamos comparar a quantidade de registros existentes no CSV com a quantidade armazenada no PostgreSQL.

In [ ]:
# ==========================================================
# ETAPA 8 - CARGA DOS DADOS NA CAMADA BRONZE
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    # Abre uma nova conexão com o PostgreSQL utilizando
    # as configurações definidas anteriormente.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    # Cria o cursor responsável por executar
    # os comandos SQL.
    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. LIMPAR A TABELA
    # ------------------------------------------------------

    # TRUNCATE remove todos os registros existentes
    # na tabela sem remover sua estrutura.
    #
    # Isso permite executar novamente esta célula
    # sem duplicar os dados.
    cursor.execute("""
        TRUNCATE TABLE bronze.vendas;
    """)


    # ------------------------------------------------------
    # 3. DEFINIR O COMANDO INSERT
    # ------------------------------------------------------

    # Criamos o comando SQL responsável pela inserção.
    #
    # Os %s funcionam como placeholders.
    # Os valores reais serão enviados separadamente
    # pelo psycopg2.
    sql_insert = """
        INSERT INTO bronze.vendas (
            id_venda,
            data_venda,
            cliente,
            produto,
            categoria,
            regiao,
            quantidade,
            valor_unitario,
            desconto_percentual,
            valor_total,
            forma_pagamento
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """


    # ------------------------------------------------------
    # 4. PERCORRER O DATAFRAME
    # ------------------------------------------------------

    # itertuples() percorre o DataFrame linha por linha.
    #
    # index=False:
    # não envia o índice criado pelo Pandas.
    #
    # name=None:
    # retorna cada registro como uma tupla simples.
    for linha in df.itertuples(index=False, name=None):

        # Executa o INSERT utilizando os valores
        # presentes na linha atual.
        cursor.execute(sql_insert, linha)


    # ------------------------------------------------------
    # 5. CONFIRMAR A TRANSAÇÃO
    # ------------------------------------------------------

    # commit() confirma todas as inserções realizadas.
    conexao.commit()

    print("✅ Carga da camada Bronze concluída!")
    print("Registros enviados:", len(df))


except Exception as erro:

    # ------------------------------------------------------
    # 6. TRATAMENTO DE ERROS
    # ------------------------------------------------------

    # Caso ocorra algum problema durante a carga,
    # desfazemos a transação.
    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro durante a carga da camada Bronze.")
    print("Tipo do erro:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    # ------------------------------------------------------
    # 7. ENCERRAR OS RECURSOS
    # ------------------------------------------------------

    # Fecha o cursor caso tenha sido criado.
    if "cursor" in locals():
        cursor.close()

    # Fecha a conexão com o banco.
    if "conexao" in locals():
        conexao.close()

## Etapa 9 — Validação da carga

Uma carga de dados não deve ser considerada concluída apenas porque o código foi executado sem erros.

Precisamos verificar se a quantidade de registros armazenada no destino corresponde à quantidade existente na fonte.

Vamos comparar:

**Fonte:** quantidade de registros do DataFrame.

**Destino:** quantidade de registros da tabela `bronze.vendas`.

Se os dois valores forem iguais, nossa primeira validação será aprovada.

In [ ]:
# ==========================================================
# ETAPA 9 - VALIDAÇÃO DA CARGA
# ==========================================================

try:

    # Abre novamente a conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # COUNT(*) conta todos os registros existentes
    # na tabela bronze.vendas.
    cursor.execute("""
        SELECT COUNT(*)
        FROM bronze.vendas;
    """)


    # fetchone() recupera uma linha do resultado.
    #
    # [0] acessa o primeiro valor dessa linha,
    # que corresponde ao resultado do COUNT(*).
    quantidade_bronze = cursor.fetchone()[0]


    # A quantidade da fonte vem do DataFrame.
    quantidade_fonte = len(df)


    print("Registros na fonte:", quantidade_fonte)
    print("Registros na Bronze:", quantidade_bronze)


    # Compara fonte e destino.
    if quantidade_fonte == quantidade_bronze:
        print("✅ VALIDAÇÃO APROVADA: fonte e Bronze possuem a mesma quantidade.")
    else:
        print("❌ VALIDAÇÃO REPROVADA: existem diferenças entre fonte e Bronze.")


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 9 — Validação da Camada Bronze

Os dados foram inseridos no PostgreSQL, mas isso não significa automaticamente que a carga está correta.

Em Engenharia de Dados, devemos validar o resultado do processo.

Nesta etapa vamos verificar:

1. Quantidade de registros na fonte e no destino;
2. Quantidade de IDs distintos;
3. Existência de valores nulos;
4. Faturamento total da fonte e da Bronze.

A ideia é comparar a **fonte (CSV)** com o **destino (PostgreSQL)**.

Se os principais indicadores forem iguais, teremos uma evidência de que os dados foram carregados corretamente.

In [ ]:
# ==========================================================
# ETAPA 9 - VALIDAÇÃO DA CAMADA BRONZE
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    # Cria uma nova conexão com o banco PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    # Cria o cursor utilizado para executar SQL.
    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. CONSULTAR A CAMADA BRONZE
    # ------------------------------------------------------

    # COUNT(*) conta todos os registros.
    #
    # COUNT(DISTINCT id_venda) conta quantos IDs
    # diferentes existem.
    #
    # SUM(valor_total) calcula o faturamento total
    # armazenado na camada Bronze.
    cursor.execute("""
        SELECT
            COUNT(*) AS total_registros,
            COUNT(DISTINCT id_venda) AS ids_distintos,
            SUM(valor_total) AS faturamento_total
        FROM bronze.vendas;
    """)


    # fetchone() recupera a linha retornada pela consulta.
    resultado = cursor.fetchone()


    # Cada posição corresponde a uma coluna do SELECT.
    registros_bronze = resultado[0]
    ids_bronze = resultado[1]
    faturamento_bronze = float(resultado[2])


    # ------------------------------------------------------
    # 3. CALCULAR OS MESMOS INDICADORES NA FONTE
    # ------------------------------------------------------

    # len(df) retorna a quantidade de linhas do CSV.
    registros_fonte = len(df)

    # nunique() conta os valores únicos da coluna id_venda.
    ids_fonte = df["id_venda"].nunique()

    # sum() soma o valor_total da fonte.
    faturamento_fonte = df["valor_total"].sum()


    # ------------------------------------------------------
    # 4. EXIBIR A COMPARAÇÃO
    # ------------------------------------------------------

    print("=== VALIDAÇÃO BRONZE ===\n")

    print("Registros")
    print("Fonte:", registros_fonte)
    print("Bronze:", registros_bronze)

    print("\nIDs únicos")
    print("Fonte:", ids_fonte)
    print("Bronze:", ids_bronze)

    print("\nFaturamento")
    print(f"Fonte:  R$ {faturamento_fonte:.2f}")
    print(f"Bronze: R$ {faturamento_bronze:.2f}")


finally:

    # ------------------------------------------------------
    # 5. ENCERRAR CONEXÃO
    # ------------------------------------------------------

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 10 — Validação de Qualidade da Camada Bronze

Após validar que a quantidade de registros, os IDs e os valores financeiros chegaram corretamente ao PostgreSQL, vamos realizar algumas verificações básicas de qualidade.

Nesta etapa verificaremos:

- valores nulos;
- IDs duplicados;
- quantidades inválidas;
- valores negativos;
- descontos fora do intervalo esperado.

Essas verificações não têm como objetivo corrigir os dados.

A camada Bronze preserva os dados recebidos da fonte.

Os problemas encontrados serão tratados posteriormente na camada **Silver**.

In [ ]:
# ==========================================================
# ETAPA 10 - VALIDAÇÃO DE QUALIDADE DA CAMADA BRONZE
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    # Cria uma conexão com o banco PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    # Cria o cursor que será responsável
    # por executar os comandos SQL.
    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. VERIFICAR IDs DUPLICADOS
    # ------------------------------------------------------

    # COUNT(*) conta todos os registros.
    #
    # COUNT(DISTINCT id_venda) conta apenas
    # os identificadores diferentes.
    #
    # A diferença entre os dois valores indica
    # quantos registros possuem IDs repetidos.
    cursor.execute("""
        SELECT
            COUNT(*) - COUNT(DISTINCT id_venda)
        FROM bronze.vendas;
    """)

    ids_duplicados = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 3. VERIFICAR VALORES NULOS
    # ------------------------------------------------------

    # Aqui verificamos se alguma das principais
    # informações da venda está ausente.
    cursor.execute("""
        SELECT COUNT(*)
        FROM bronze.vendas
        WHERE
            id_venda IS NULL
            OR data_venda IS NULL
            OR cliente IS NULL
            OR produto IS NULL
            OR categoria IS NULL
            OR regiao IS NULL
            OR quantidade IS NULL
            OR valor_unitario IS NULL
            OR desconto_percentual IS NULL
            OR valor_total IS NULL
            OR forma_pagamento IS NULL;
    """)

    registros_nulos = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 4. VERIFICAR QUANTIDADES INVÁLIDAS
    # ------------------------------------------------------

    # Uma venda não deveria possuir quantidade
    # igual ou inferior a zero.
    cursor.execute("""
        SELECT COUNT(*)
        FROM bronze.vendas
        WHERE quantidade <= 0;
    """)

    quantidades_invalidas = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 5. VERIFICAR VALORES NEGATIVOS
    # ------------------------------------------------------

    # Verifica preços ou valores totais negativos.
    cursor.execute("""
        SELECT COUNT(*)
        FROM bronze.vendas
        WHERE
            valor_unitario < 0
            OR valor_total < 0;
    """)

    valores_negativos = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 6. VERIFICAR DESCONTOS INVÁLIDOS
    # ------------------------------------------------------

    # Consideramos válido um desconto entre
    # 0% e 100%.
    cursor.execute("""
        SELECT COUNT(*)
        FROM bronze.vendas
        WHERE
            desconto_percentual < 0
            OR desconto_percentual > 100;
    """)

    descontos_invalidos = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 7. EXIBIR O RELATÓRIO
    # ------------------------------------------------------

    print("=== QUALIDADE DA CAMADA BRONZE ===\n")

    print("IDs duplicados:", ids_duplicados)
    print("Registros com valores nulos:", registros_nulos)
    print("Quantidades inválidas:", quantidades_invalidas)
    print("Valores negativos:", valores_negativos)
    print("Descontos inválidos:", descontos_invalidos)


finally:

    # ------------------------------------------------------
    # 8. ENCERRAR OS RECURSOS
    # ------------------------------------------------------

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()